# video-app lab 02 - FIRST REAL CLIP (Wan2.1 T2V-1.3B)
Runtime: **T4 GPU** (verified: 14.6 GB VRAM, 51 GB RAM, 193 GB disk).
Free, open-source, Apache-2.0. No API key, no card. Time on T4: expect 10-40 min.
Cached weights make re-runs faster after the first time.


In [ ]:
!pip -q install "diffusers>=0.33" transformers accelerate safetensors sentencepiece protobuf ftfy imageio imageio-ffmpeg
import torch, diffusers, transformers
print('diffusers', diffusers.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)


In [ ]:
import time, torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
from IPython.display import Video, display

MODEL = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
DTYPE = torch.float16   # T4: fp16 is the fast path

t0 = time.time()
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('loaded in', round(time.time()-t0), 's')
print('gpu mem used GB:', round(torch.cuda.memory_allocated()/1024**3, 2))


In [ ]:
PROMPT = ('A red fox trotting through fresh snow in a pine forest, morning light, '
          'cinematic, shallow depth of field, steady camera')
NEG = 'blurry, low quality, distorted, watermark, text, jitter'
H, W, FRAMES, STEPS = 480, 640, 33, 20   # 33 frames = ~2 s at 16 fps

torch.manual_seed(42)
t0 = time.time()
out = pipe(prompt=PROMPT, negative_prompt=NEG, height=H, width=W,
           num_frames=FRAMES, num_inference_steps=STEPS, guidance_scale=5.0).frames[0]
secs = round(time.time()-t0)
path = '/content/clip_01_fox.mp4'
export_to_video(out, path, fps=16)
print('GENERATED in', secs//60, 'min', secs%60, 's ->', path)
print('peak gpu GB:', round(torch.cuda.max_memory_allocated()/1024**3, 2))

import os
print('file MB:', round(os.path.getsize(path)/1024**2, 1))
display(Video(path, embed=True, width=480))
files.download(path) if 'files' in dir() else None


### If it fails
- `CUDA out of memory` -> lower H/W to 384/512 and FRAMES to 17, re-run only the last cell.
- `bf16 not supported` -> keep DTYPE float16.
- Slow is fine: record the minutes, that is our real per-clip cost on free hardware.
